# Lung dataset selection

Nowy narząd, ta sama metodyka co `brain_dataset.ipynb`: `msi_dataset_manager.exploration.DatasetExplorer` + `DatasetReview`/`DatasetReviewProfile` (`packages/msi_dataset_manager/src/msi_dataset_manager/exploration/dataset_review.py`) do obiektywnej detekcji duplikatów, wariantów QC i heurystyki morfologii. Zero zmian w bibliotece, zero pobrań surowych danych — wynikiem jest wyłącznie przejrzana lista kandydatów wyeksportowana jako `filter.json`/`selection.json`.

**Uwaga na wielkość puli:** przy `organism=Mouse, polarity=Negative` (ten sam profil technologiczny co kidney/liver/brain) pula to **29** kandydatów — dużo mniej niż surowa liczba 809 z METASPACE (ta obejmuje wszystkie organizmy i obie polaryzacje). To nie jest błąd filtra, tylko rzeczywisty rozmiar części katalogu spójnej technologicznie z resztą projektu.

**Zakres m/z: `mz_min=200, mz_max=900`, przyjęty jako domyślny punkt startowy spójny z kidney** (patrz `brain_dataset.ipynb`, sekcja 5, gdzie ten wariant wypadł najlepiej na dostępnym katalogu). To NIE jest wynik osobnej optymalizacji dla tego narządu — ujednolicenie zakresu m/z między wszystkimi narządami jest świadomie odłożone na później (tak jak w `kidney_dataset_repaired.ipynb`/`liver_dataset_repaired.ipynb`). Poniżej pokazuję explicité, ile kandydatów ten zakres realnie pokrywa, żebyś widział koszt tego wyboru dla tego konkretnego narządu.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import re

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer, DatasetReviewProfile

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów

Ten sam filtr biologiczny co `brain_dataset.ipynb`: `condition=["Wildtype", "Wtype", "N/A"]`, `organism=Mouse`, `polarity=Negative`. Bez `mz_min`/`mz_max` na tym etapie, żeby audyt duplikatów/jakości objął całą pulę, nie tylko to, co już pasuje do docelowego zakresu.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Lung",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "condition", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 29 datasets


,dataset_id,name,condition,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-07-30_14h53m30s,20260121-f1-9aa-3-root mean square,N/A,Orbitrap,DESI,57.006265,998.735944,1071
1,2026-07-30_14h51m23s,20260121-f1-9aa-2-root mean square,N/A,Orbitrap,DESI,64.360075,999.615261,1531
2,2026-07-30_14h54m59s,20260121-f1-9aa-4-root mean square,N/A,Orbitrap,DESI,57.005940,999.736569,2256
3,2022-03-24_16h56m33s,Lung_tissue,Wildtype,timsTOF Flex,AP-MALDI,75.003750,799.980000,64456
4,2023-09-07_11h00m06s,lung10um_bndm10mg_neg5um-root mean square,Wildtype,timsTOF fleX,MALDI,75.001875,999.985000,261477
5,2023-07-27_15h14m41s,mouselung_metneg_10um,Wildtype,timsTOF fleX,MALDI,75.001500,999.985000,22951
6,2023-09-07_09h18m48s,lung10um_bndm5mg_neg50um-root mean square,Wildtype,timsTOF fleX,MALDI,75.001500,999.985000,4245
7,2023-09-07_08h16m31s,lung_bndm10mg_neg_20um-root mean square,Wildtype,timsTOF fleX,MALDI,75.001500,999.985000,33592
8,2023-09-07_06h10m42s,lung_10um_9aasub_neg-root mean square,Wildtype,timsTOF fleX,MALDI,75.001875,999.985000,26773
9,2023-09-07_05h52m50s,lung_10um_9aa_neg-root mean square,Wildtype,timsTOF fleX,MALDI,75.002250,999.985000,15294


## 2. Ręczna kontrola jakości, której żadna reguła biblioteczna nie łapie

Ten sam skan co w `kidney_dataset_repaired.ipynb`: niedopasowanie gatunku/tkanki w nazwie, jawne oznaczenia testowe.

In [4]:
suspect_pattern = r"zebrafish|drosophila|\brat\b|\bhuman\b|\btest\b|\(test\)|calib|standard"
suspects = results[results["name"].str.contains(suspect_pattern, case=False, na=False, regex=True)]
display(suspects[["dataset_id", "name", "condition", "organisms", "pixel_count"]] if len(suspects) else "none found")

'none found'

## 3. Przegląd biblioteczny (`DatasetExplorer.review_current`)

Brak wbudowanego profilu `"lung"` w `_PROFILES` (tylko `brain`/`liver`) — przekazuję `DatasetReviewProfile` z poziomu notebooka, tak jak w `kidney_dataset_repaired.ipynb`.

In [5]:
lung_profile = DatasetReviewProfile(
    low_pixel_threshold=1000,  # minimum w tej puli to 1071 pikseli (kilka datasetów `20260121-f1-9aa-N-root mean square`) — próg 1000 łapie tylko to, co wyraźnie poniżej.
    morphology_pattern=r"(?:alveol|bronch|pleura|airway)",
    explicit_regional_names=frozenset(),
)
review = explorer.review_current(profile=lung_profile)

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(
    review.table.loc[
        review.table["mz_shift_qc_variant"] | review.table["morphology_hint"].eq("regional_or_microregion") | review.table["low_pixel_flag"],
        ["dataset_id", "name", "pixel_count", "mz_shift_qc_variant", "morphology_hint", "low_pixel_flag"],
    ]
)

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,0
1,mz_shift_qc_variants,0
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id
26,technical-0006,2017-06-02_13h26m48s,2016_05_17_crackit_d7_p1_B_Processed_negative,9875,ambiguous_shared_template,False,<NA>
27,technical-0006,2017-06-02_13h59m48s,2016_05_17_crackit_d7_p1_B_Processed_neg2,9875,ambiguous_shared_template,False,<NA>
28,technical-0006,2017-06-02_14h18m52s,2016_05_17_crackit_d7_p1_B_Processed,9875,ambiguous_shared_template,False,<NA>
9,technical-0009,2023-09-07_05h52m50s,lung_10um_9aa_neg-root mean square,15294,ambiguous_shared_template,False,<NA>
12,technical-0009,2023-09-06_15h05m57s,lung_10um_9aa_neg,15294,ambiguous_shared_template,False,<NA>
15,technical-0015,2023-08-09_09h14m18s,A. mouselung_ 10um_metneg_dan-total ion count,23046,ambiguous_shared_template,False,<NA>
18,technical-0015,2023-07-31_07h24m15s,mouselung_ 10um_metneg_dan-total ion count,23046,ambiguous_shared_template,False,<NA>
10,technical-0016,2023-09-07_05h43m28s,lung_8um_9aa_neg-root mean square,23400,ambiguous_shared_template,False,<NA>
13,technical-0016,2023-09-06_14h58m08s,lung_8um_9aa_neg,23400,ambiguous_shared_template,False,<NA>
8,technical-0018,2023-09-07_06h10m42s,lung_10um_9aasub_neg-root mean square,26773,ambiguous_shared_template,False,<NA>


,dataset_id,name,pixel_count,mz_shift_qc_variant,morphology_hint,low_pixel_flag


### Ręczne potwierdzenie duplikatów spoza zasięgu reguł bibliotecznych

**Sześć dodatkowych wykluczeń, które `review_current` poprawnie oznaczył jako `ambiguous_shared_template`, ale które ręcznie potwierdzam jako duplikaty** — dowód jest przytłaczający (identyczny `pixel_count` **i** identyczne `mz_min`/`mz_max` co do kilku miejsc po przecinku), tylko konwencja nazewnictwa w tym labie (myślnik bez spacji, prefiks 'A. ' jako numeracja) nie pasuje do tokenów rozpoznawanych przez `dataset_review.py`. To **nie** jest błąd biblioteki — to dokładnie ten rodzaj przypadku, który `ambiguous_shared_template` ma poprawnie zostawiać do ręcznego przeglądu, a nie fałszywie wykluczać automatycznie.

In [6]:
MANUAL_DUPLICATE_EXCLUSIONS = {
    "2017-06-02_13h26m48s": (
        "identyczny pixel_count (9875) i mz_min/mz_max (do 6. miejsca po przecinku) jak '2016_05_17_crackit_d7_p1_B_Processed' i '..._neg2' -- ten sam surowy skan zgłoszony 3x pod różnymi nazwami (residual-check tego nie złapał, bo różnice w nazwie to 'negative'/'neg2'/nic, nie rozpoznany token techniczny)"
    ),
    "2017-06-02_13h59m48s": (
        "patrz wyżej -- ten sam klaster 3-way, zachowuję tylko '2016_05_17_crackit_d7_p1_B_Processed'"
    ),
    "2023-09-07_06h10m42s": (
        "identyczny pixel_count/mz jak 'lung_10um_9aasub_neg' -- to sam skan, sufiks '-root mean square' bez spacji przed myślnikiem nie pasuje do wzorca ' - root mean square$' w bibliotece, więc review nie połączył residuali automatycznie"
    ),
    "2023-09-07_05h52m50s": (
        "identyczny pixel_count/mz jak 'lung_10um_9aa_neg' -- ten sam problem formatowania myślnika co wyżej"
    ),
    "2023-09-07_05h43m28s": (
        "identyczny pixel_count/mz jak 'lung_8um_9aa_neg' -- ten sam problem formatowania myślnika co wyżej"
    ),
    "2023-08-09_09h14m18s": (
        "identyczny pixel_count (23046) i mz jak 'mouselung_ 10um_metneg_dan-total ion count' -- różnica to tylko prefiks 'A. ' (numeracja listy), nie token techniczny"
    ),
}
display(results.loc[results["dataset_id"].isin(MANUAL_DUPLICATE_EXCLUSIONS), ["dataset_id", "name", "pixel_count"]])
explorer.exclude(list(MANUAL_DUPLICATE_EXCLUSIONS))

,dataset_id,name,pixel_count
8,2023-09-07_06h10m42s,lung_10um_9aasub_neg-root mean square,26773
9,2023-09-07_05h52m50s,lung_10um_9aa_neg-root mean square,15294
10,2023-09-07_05h43m28s,lung_8um_9aa_neg-root mean square,23400
15,2023-08-09_09h14m18s,A. mouselung_ 10um_metneg_dan-total ion count,23046
26,2017-06-02_13h26m48s,2016_05_17_crackit_d7_p1_B_Processed_negative,9875
27,2017-06-02_13h59m48s,2016_05_17_crackit_d7_p1_B_Processed_neg2,9875


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-07-30_14h53m30s,20260121-f1-9aa-3-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2026-07-30_14...,Mus musculus (mouse),Lung,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-07-30_14h51m23s,20260121-f1-9aa-2-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2026-07-30_14...,Mus musculus (mouse),Lung,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-07-30_14h54m59s,20260121-f1-9aa-4-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2026-07-30_14...,Mus musculus (mouse),Lung,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
3,2022-03-24_16h56m33s,Lung_tissue,metaspace,None,https://metaspace2020.eu/dataset/2022-03-24_16...,Mus musculus (mouse),Lung,Wildtype,animal,,...,None,None,0.1,None,None,None,None,None,,False
4,2023-09-07_11h00m06s,lung10um_bndm10mg_neg5um-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2023-09-07_11...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False
5,2023-07-27_15h14m41s,mouselung_metneg_10um,metaspace,None,https://metaspace2020.eu/dataset/2023-07-27_15...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False
6,2023-09-07_09h18m48s,lung10um_bndm5mg_neg50um-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2023-09-07_09...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False
7,2023-09-07_08h16m31s,lung_bndm10mg_neg_20um-root mean square,metaspace,None,https://metaspace2020.eu/dataset/2023-09-07_08...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False
8,2023-09-06_15h29m14s,lung_10um_9aasub_neg,metaspace,None,https://metaspace2020.eu/dataset/2023-09-06_15...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False
9,2023-09-06_15h05m57s,lung_10um_9aa_neg,metaspace,None,https://metaspace2020.eu/dataset/2023-09-06_15...,Mouse,Lung,Wildtype,Navie,,...,None,None,0.1,None,None,None,None,None,,False


## 4. Zastosowanie reguł i finalna lista

`high_confidence_duplicates` + `mz_shift_qc_variants` + `explicit_regional_fragments` (wszystkie trzy, dla spójności z pozostałymi notebookami, nawet gdy akurat wychodzi 0). `morphology_hint`/`low_pixel_flag` zostają doradczo.

In [7]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Lung",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from steps 2/3/4 persist across this re-query.
results_lung = explorer.filter(final_filters)
print(f"final lung shortlist: {len(results_lung)} datasets (of {len(results)} broad candidates)")
display(results_lung[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

final lung shortlist: 15 datasets (of 29 broad candidates)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2026-07-30_14h53m30s,20260121-f1-9aa-3-root mean square,Orbitrap,1071,2,0
1,2026-07-30_14h51m23s,20260121-f1-9aa-2-root mean square,Orbitrap,1531,8,3
2,2026-07-30_14h54m59s,20260121-f1-9aa-4-root mean square,Orbitrap,2256,4,2
3,2023-09-07_11h00m06s,lung10um_bndm10mg_neg5um-root mean square,timsTOF fleX,261477,190,6
4,2023-07-27_15h14m41s,mouselung_metneg_10um,timsTOF fleX,22951,42,31
5,2023-09-07_09h18m48s,lung10um_bndm5mg_neg50um-root mean square,timsTOF fleX,4245,288,35
6,2023-09-07_08h16m31s,lung_bndm10mg_neg_20um-root mean square,timsTOF fleX,33592,219,5
7,2023-09-06_15h29m14s,lung_10um_9aasub_neg,timsTOF fleX,26773,182,2
8,2023-09-06_15h05m57s,lung_10um_9aa_neg,timsTOF fleX,15294,14,1
9,2023-09-06_14h58m08s,lung_8um_9aa_neg,timsTOF fleX,23400,7,0


## 5. Grupowanie w serie biologiczne (Poziom 3)

Ta sama heurystyka co w `brain_dataset.ipynb` — tylko do identyfikacji grup, które muszą zostać razem w tym samym podziale train/validation/test, nie do automatycznego wybierania reprezentanta.

In [8]:
def biological_series_key(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"_\d+ppm$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"_replicate\d+$", "", s)
    s = re.sub(r"_s\d+$", "", s)
    s = re.sub(r"_\d+$", "", s)
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s


results_lung["biological_series_id"] = results_lung["name"].apply(biological_series_key)
n_series = results_lung["biological_series_id"].nunique()
print(f"final shortlist: {len(results_lung)} datasets across {n_series} name-derived series")
series_sizes = results_lung.groupby("biological_series_id").size().sort_values(ascending=False)
display(series_sizes[series_sizes > 1])

final shortlist: 15 datasets across 15 name-derived series


Series([], dtype: int64)

In [9]:
output_path = Path("data/lung_workspace/configs/datasets/lung")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/lung_workspace/configs/datasets/lung/filter.json'),
 'selection': PosixPath('data/lung_workspace/configs/datasets/lung/selection.json')}

## Podsumowanie

- Szeroka pula (`Mouse`+`Negative`+`Wildtype`/`Wtype`/`N/A`): patrz sekcja 1 dla dokładnej liczby.
- Kontrola jakości i przegląd biblioteczny: sekcje 2–3.
- Zakres m/z `200–900` przyjęty jako domyślny, spójny z kidney — **nie zoptymalizowany osobno dla tego narządu**, patrz zastrzeżenie na początku notebooka.
- Eksport do `data/lung_workspace/configs/datasets/lung/` — pierwszy raz dla tego narządu, brak wcześniejszej konfiguracji do porównania.
- Analiza wspólnego zakresu m/z między wszystkimi narządami — świadomie pominięta, do rozwiązania osobno.